In [3]:
import pandas as pd
import numpy as np
import random

In [7]:
def get_entropy(y: pd.Series):
    unique_labels = y.unique()
    
    if len(unique_labels) == 1:
        return 0
    
    n = len(y)
    count_1 = y.sum()
    count_0 = n - count_1

    proba_0 = count_0 / n
    proba_1 = count_1 / n

    return - (np.log2(proba_0) * proba_0 + np.log2(proba_1) * proba_1)

def get_gini(y: pd.Series):
    unique_labels = y.unique()

    if len (unique_labels) == 1:
        return 0

    n = len(y)
    count_1 = y.sum()
    count_0 = n - count_1

    proba_0 = count_0 / n
    proba_1 = count_1 / n

    return 1 - proba_0 ** 2 - proba_1 ** 2
 
class Leaf:
    def __init__(self, parent, samples, proba, depth):
        self.parent = parent
        self.samples = samples
        self.proba = proba
        self.depth = depth

    def __str__(self):
        return f'{"  " * self.depth} {self.proba}'

class Node:
    def __init__(self, parent=None):
        self.parent = parent
        self.left = None
        self.right = None
        self.samples = None
        self.depth = 1
        self.split_value = None
        self.column = None
        self.ig = None
        self.is_left = None

    def __str__(self):
        return  '\n'.join([
        f'{"  " * self.depth} {self.column} {self.split_value}',
        f'{self.left}',
        f'{self.right}'])

    def predict_proba(self, row: pd.Series):
        if row.loc[self.column] <= self.split_value:
            if isinstance(self.left, Leaf):
                return self.left.proba
            else: 
                return self.left.predict_proba(row)
        else:
            if isinstance(self.right, Leaf):
                return self.right.proba
            else: 
                return self.right.predict_proba(row)

class MyTreeClf:
    def __init__(self, max_depth: int = 5, min_samples_split: int = 2,
                 max_leafs: int = 20, bins=None, criterion="entropy", N=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_leafs = max_leafs
        self.nodes_cnt = 1
        self.leafs_cnt = None
        self.bins = bins
        self.split_values = None
        self.criterion = criterion
        self.fi = dict()
        self.N = N

    def __str__(self):
        return f"MyTreeClf class: max_depth={self.max_depth}, min_samples_split={self.min_samples_split}, max_leafs={self.max_leafs}"

    def __build_split_values(self, X):
        self.split_values = dict()
        for column in X.columns:
            unique_values = X[column].unique()
            if (len(unique_values) - 1 <= self.bins - 1):
                sorted_unique_values = np.sort(unique_values)
                self.split_values[column] = (sorted_unique_values[:-1] + sorted_unique_values[1:]) / 2
            else:
                count_elements, split_values = np.histogram(X[column], bins = self.bins)
                self.split_values[column] = split_values[1:-1]

    def get_split_values(self, X, column):
        if self.bins is None:
            unique_values = X[column].unique()
            sorted_unique_values = np.sort(unique_values)
            return (sorted_unique_values[:-1] + sorted_unique_values[1:]) / 2
        else:
            max_value = X[column].max()
            min_value = X[column].min()
            return self.split_values[column][ (min_value <= self.split_values[column]) & (max_value > self.split_values[column])]

    def __init_fi(self, X):
        for column in X.columns:
            self.fi[column] = 0 
    
    def fit(self, X: pd.DataFrame, y: pd.Series):
        self.root = Node()
        self.__init_fi(X)
        self.N = len(y) if self.N is None else self.N
        if self.bins is not None:
            self.__build_split_values(X)
            
        self.build_tree_recursive(self.root, X, y)
        self.leafs_cnt = self.nodes_cnt + 1

    def get_best_split(self, X: pd. DataFrame, y : pd.Series):

        igs = []
        
        for column in X.columns:
            split_values = self.get_split_values(X, column)
            if self.criterion == "entropy":
                criterion_function = get_entropy
            elif self.criterion == "gini":
                criterion_function = get_gini
            criterion = criterion_function(y)
            
            for split_value in split_values: 

                left_indexes = X.index[X[column] <= split_value]
                left_y = y.loc[left_indexes]
                left_criterion = criterion_function(left_y)
                
                right_indexes = X.index[X[column] > split_value]
                right_y = y.loc[right_indexes]
                right_criterion = criterion_function(right_y)
    
                ig = criterion - left_criterion * len(left_indexes) / len(y) - right_criterion * len(right_indexes) / len(y) 
    
                igs.append((column, split_value, ig))
        if igs:    
            return max(igs, key= lambda x: x[2])
        else:
            return None, None, None

    
    def predict_proba(self, X: pd.DataFrame) -> pd.Series:
        probas = []
        for index, row in X.iterrows():
            proba = self.root.predict_proba(row)
            probas.append(proba)
        return pd.Series(probas, X.index)

    def predict(self, X: pd.DataFrame) -> pd.Series:
        probas = self.predict_proba(X)
        return probas > 0.5
        
    def build_tree_recursive(self, node: Node, X, y):

        node.column, node.split_value, node.ig = self.get_best_split(X, y)
        
        if node.column is not None:
            self.fi[node.column] += len(y) / self.N * node.ig
            
            left_indexes = X.index[X[node.column] <= node.split_value]
            left_cnt = len(left_indexes)
            left_y = y.loc[left_indexes]
            
            if len(left_y.unique()) == 1 or left_cnt < self.min_samples_split or \
                self.nodes_cnt + 1 >= self.max_leafs or node.depth == self.max_depth \
                :
                node.left = Leaf(node, len(left_y), left_y.mean(), node.depth + 1)
            else:
                node.left = Node(node)
                node.left.depth = node.depth + 1
                node.left.is_left = True
                self.nodes_cnt += 1
                self.build_tree_recursive(node.left, X.loc[left_indexes], left_y)
    
            right_indexes = X.index[X[node.column] > node.split_value]
            right_cnt = len(right_indexes)
            right_y = y.loc[right_indexes]
            
            if len(right_y.unique()) == 1  or right_cnt < self.min_samples_split  or \
                self.nodes_cnt + 1 >= self.max_leafs or node.depth == self.max_depth:
                node.right = Leaf(node, len(right_y), right_y.mean(), node.depth + 1)
            else:
                node.right = Node(node)
                node.right.depth = node.depth + 1
                node.right.is_left = False
                self.nodes_cnt += 1
                self.build_tree_recursive(node.right, X.loc[right_indexes], right_y)
        else:
            parent = node.parent
            if node.is_left:
                parent.left = Leaf(parent, len(y), y.mean(), parent.depth + 1)
            else:
                parent.right = Leaf(parent, len(y), y.mean(), parent.depth + 1)
            self.nodes_cnt -= 1

class MyForestClf:
    def __init__(self, n_estimators = 10, max_features = 0.5,
                 max_samples = 0.5, random_state = 42,
                 max_depth = 5, min_samples_split = 2,
                 max_leafs = 20, bins = 16,
                 criterion = "entropy",
                 oob_score = None
                 ):
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.max_samples = max_samples
        self.random_state = random_state
        self.max_depth = max_depth
        self.min_samples_split = 2
        self.max_leafs = max_leafs
        self.bins = bins
        self.criterion = criterion
        self.leafs_cnt = 0
        self.trees = []
        self.fi = dict()
        self.oob_score = oob_score
        self.oob_score_ = []

    def __str__(self):
        return f"MyForestClf class: n_estimators={self.n_estimators}, max_features={self.max_features},"\
               f" max_samples={self.max_samples}, max_depth={self.max_depth},"\
               f" min_samples_split={self.min_samples_split}, max_leafs={self.max_leafs},"\
               f" bins={self.bins}, criterion={self.criterion}, random_state={self.random_state}"

    def _accuracy(self, y_true, y_pred) -> float:
        return np.mean(y_true == y_pred)

    def _precision(self, y_true, y_pred) -> float:
        mask = y_pred == 1
        return y_true[mask].mean()

    def _recall(self, y_true, y_pred) -> float:
        mask = y_true == 1
        return y_pred[mask].mean()

    def _f1(self, y_true, y_pred) -> float:
        precision = self._precision(y_true, y_pred)
        recall = self._recall(y_true, y_pred)
        f1 = 2 * precision * recall / (precision + recall)
        return f1

    def _roc_auc(self, y_true: pd.Series, y_pred: pd.Series):
        y_true = y_true.copy()
        y_true.name = 'true'
        y_pred = y_pred.copy()
        y_pred.name = 'pred'
        y_pred = np.round(y_pred, 10)
        
        y = pd.concat([y_pred, y_true], axis=1).sort_values(["pred", "true"], ascending=False)    

        roc_auc = 0 
        for i in range(len(y)):
            if y['true'].iloc[i] == 0:
                p = y['pred'].iloc[i]
                for j in range(i):
                    if y['true'].iloc[j] == 1 and y['pred'].iloc[j] > p :
                        roc_auc += 1
                    elif y['true'].iloc[j] == 1 and y['pred'].iloc[j] == p:
                        roc_auc += 0.5
        roc_auc /= np.sum(y['true']) * (len(y['true']) - np.sum(y['true']))

        return roc_auc
    
    
    def fit(self, X: pd.DataFrame, y: pd.Series):
        random.seed(self.random_state)

        for column in X.columns:
            self.fi[column] = 0

        if self.oob_score is not None:
            oob_ys = []

        for i in range(self.n_estimators):
        
            cols_smpl_cnt = round(X.shape[1] * self.max_features)
            cols_idx = random.sample(X.columns.to_list(), cols_smpl_cnt)
    
            rows_smpl_cnt = round(X.shape[0] * self.max_samples)
            rows_idx = random.sample(X.index.to_list(), rows_smpl_cnt)
            
            if self.oob_score is not None:
                set_rows_idx = set(rows_idx)
                oob_indexes = X.index.map(lambda x : not (x in set_rows_idx))
                oob_X = X[cols_idx].loc[oob_indexes]
                oob_y = y.loc[oob_indexes]
            
            my_tree_reg = MyTreeClf(self.max_depth, self.min_samples_split, self.max_leafs, self.bins, self.criterion, len(X))

            my_tree_reg.fit(X[cols_idx].loc[rows_idx], y.loc[rows_idx])

            if self.oob_score is not None:

                oob_y_predict = my_tree_reg.predict_proba(oob_X)
                oob_ys.append(oob_y_predict)
                 
            self.trees.append(my_tree_reg)

            self.leafs_cnt += my_tree_reg.leafs_cnt
            
        if self.oob_score is not None:
            y_matrix = pd.concat(oob_ys, axis=1)
            y_mean = y_matrix.mean(axis=1)
                
            if self.oob_score == 'accuracy':
                metric = self._accuracy
            if self.oob_score == 'precision':
                metric = self._precision
            if self.oob_score == 'recall': 
                metric = self._recall
            if self.oob_score == 'f1':
                metric = self._f1
            if self.oob_score == 'roc_auc':
                metric = self._roc_auc
            
            indexes = y_mean.index[y_mean.notnull()]
            if self.oob_score == 'roc_auc':
                self.oob_score_ = metric(y.loc[indexes], y_mean.loc[indexes])
            else:
                self.oob_score_ = metric(y.loc[indexes], (y_mean > 0.5).loc[indexes])
                                         
        for tree in self.trees:
            for column in X.columns:
                self.fi[column] += tree.fi.get(column, 0)
    
    def predict(self, X, type):
        y_s = []
        for tree in self.trees:
            if type == "mean":
                y_s.append(tree.predict_proba(X))

            if type == "vote":
                y_s.append(tree.predict(X))
        
        y_s = pd.concat(y_s, axis=1)

        if type == "mean":
            return y_s.mean(axis=1) > 0.5       

        if type == "vote":
            return y_s.mean(axis=1) >= 0.5

    def predict_proba(self, X):
        y_s = []
        for tree in self.trees:
            y_s.append(tree.predict_proba(X))

        y_s = pd.concat(y_s, axis=1)
        return y_s.mean(axis=1)